In [1]:
#Importo Pacchetti Necessari
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import  MinMaxScaler, OneHotEncoder
from sklearn.metrics import log_loss,accuracy_score, roc_auc_score, silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV 
import requests

#par in input DataSet, ID, score, statistical severity (incide su min_lift e parametri di DBSCAN e PCA, Grid Search vs random search, MA - Alta - Moderata - Sufficiente), threshold (numero da 1 a 99 che identifica il percentile di  propensione)

In [2]:

# ─────────────────────────────────────────────
# 1. CONFIGURAZIONE
# ─────────────────────────────────────────────

# Indirizzo locale di Ollama (non cambiare)
OLLAMA_URL = "http://localhost:11434/api/chat"

# Modello da usare — deve essere già scaricato con "ollama pull <nome>"
MODELLO = "phi3:mini"   # cambia con "llama3" o "phi3" se preferisci

# Parola magica per uscire dal loop di feedback
PAROLA_MAGICA = "yes"

# File di output JSON con la memoria dell'agente
OUTPUT_JSON = "context_memory.json"

# ─────────────────────────────────────────────
# 2. LETTURA DEI CSV
# ─────────────────────────────────────────────

def leggi_csv(filepath: str) -> str:
    """
    Legge un file CSV e lo converte in testo tabellare
    da inserire nel prompt come contesto.
    Restituisce stringa vuota se il file non esiste.
    """
    if not os.path.exists(filepath):
        print(f"  [WARN] File non trovato: {filepath} — saltato.")
        return ""

    df = pd.read_csv(filepath, sep=";")
    return df.to_string(index=False)

In [ ]:

# ─────────────────────────────────────────────
# 3. CHIAMATA A OLLAMA (via HTTP locale)
# ─────────────────────────────────────────────

def chiedi_a_ollama(cronologia: list) -> str:
    """
    Invia la cronologia dei messaggi a Ollama in locale
    e restituisce la risposta testuale del modello.

    Ollama accetta lo stesso formato di OpenAI:
    [{"role": "user"/"assistant"/"system", "content": "..."}]
    """
    payload = {
        "model": MODELLO,
        "messages": cronologia,
        "stream": False   # risposta completa, non a token
    }

    try:
        risposta = requests.post(OLLAMA_URL, json=payload, timeout=2500)
        risposta.raise_for_status()
        dati = risposta.json()
        return dati["message"]["content"]

    except requests.exceptions.ConnectionError:
        raise ConnectionError(
            "\n❌ Ollama non è in esecuzione!\n"
            "   Avvialo con il comando: ollama serve\n"
            "   Oppure apri l'app Ollama dal menu di sistema."
        )
    except requests.exceptions.Timeout:
        raise TimeoutError(
            "\n❌ Timeout: il modello ha impiegato troppo.\n"
            "   Prova un modello più leggero come 'phi3'."
        )


# ─────────────────────────────────────────────
# 4. COSTRUZIONE DEL PROMPT INIZIALE
# ─────────────────────────────────────────────

def costruisci_prompt_iniziale(general_ctx, business_ctx, features_ctx) -> str:
    """
    Assembla il prompt con tutti e tre i livelli di contesto
    estratti dai CSV. Il modello analizzerà tutto insieme.
    """
    prompt = f"""
You are an expert agent specialized in human context analysis and data interpretation.
You are provided with three levels of contextual information extracted from CSV files.
Your task is to deeply understand these inputs and produce a structured, comprehensive summary that highlights the key insights, 
relationships, and implications for the business environment.

---
## CONTESTO GENERALE
{general_ctx if general_ctx else "Non fornito."}

---
## CONTESTO DI BUSINESS
{business_ctx if business_ctx else "Non fornito."}

---
## CONTESTO DELLE FEATURES E CONCETTI LATENTI
{features_ctx if features_ctx else "Non fornito."}

---
Based on all provided information, deliver the following:

Agent Role
Define the role the agent should assume to provide maximum value.

Focus on analytical, interpretive, and decision‑support capabilities.

General Context

Summarize the overall domain and environment.

Highlight the macro‑level dynamics, actors, and constraints.

Business Context

Describe key objectives, processes, KPIs, and business logic relevant to the scenario.

Identify strategic priorities and operational drivers.

Purpose

Specify what the agent/system must concretely accomplish.

Clarify expected outputs, decisions, or transformations.

Included Features

Explain what each variable measures and what latent concepts they represent.

Highlight how these features relate to the business problem.

Be precise, concise, and structured. Use bullet points where appropriate.
"""
    return prompt.strip()


In [ ]:

# ─────────────────────────────────────────────
# 5. ESTRAZIONE STRUTTURA JSON FINALE
# ─────────────────────────────────────────────

def estrai_struttura_json(risposta_finale: str) -> dict:
    """
    Chiede al modello di estrarre dalla risposta finale
    una struttura JSON pulita da salvare come memoria.
    """
    prompt_json = f"""
From the following analysis, extract a JSON object with the exact structure shown below.
Respond ONLY with valid JSON. And all insights and description should be prompt-designed for the next LLM agent.
No markdown, no comments, no additional text.

{{
  "role": "stringa",
  "general_context": "stringa",
  "business_context": "stringa",
  "scope": "stringa",
  "features_and_concepts": ["stringa", "stringa"]
  "other": "stringa",
}}

Analisi da cui estrarre:
{risposta_finale}
"""

    messaggi = [{"role": "user", "content": prompt_json}]
    testo = chiedi_a_ollama(messaggi)

    # Pulizia: rimuove eventuali backtick markdown che il modello aggiunge
    testo_pulito = testo.strip().strip("```json").strip("```").strip()

    try:
        struttura = json.loads(testo_pulito)
    except json.JSONDecodeError:
        # Fallback sicuro: salva la risposta grezza senza crashare
        print("  [WARN] Il modello non ha restituito JSON valido. Salvo risposta grezza.")
        struttura = {
            "role": "Non estratto automaticamente",
            "general_context": "Non estratto automaticamente",
            "business_context": "Non estratto automaticamente",
            "scope": "Non estratto automaticamente",
            "features_and_concepts": [],
            "other": "Non estratto automaticamente",
            "risposta_completa": risposta_finale
        }

    return struttura


# ─────────────────────────────────────────────
# 6. SALVATAGGIO MEMORIA IN JSON
# ─────────────────────────────────────────────

def salva_memoria(struttura: dict, filepath: str = OUTPUT_JSON):
    """
    Scrive la struttura appresa su disco in formato JSON leggibile.
    """
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(struttura, f, ensure_ascii=False, indent=2)
    print(f"\n✅ Memoria salvata in: {filepath}")


# ─────────────────────────────────────────────
# 7. LOOP PRINCIPALE DELL'AGENTE
# ─────────────────────────────────────────────

def esegui_agente_contesto(
    general_csv: str = "general_context.csv",
    business_csv: str = "business_context.csv",
    features_csv: str = "features_context.csv"
):
    """
    Funzione principale:
      1. Legge i tre CSV
      2. Analizza il contesto con Ollama
      3. Mostra la comprensione all'utente
      4. Raccoglie feedback in loop finché non arriva "yes"
      5. Estrae e salva la memoria strutturata in JSON
    """

    print("=" * 60)
    print("  AGENTE CONTESTUALE — powered by Ollama (locale)")
    print(f"  Modello: {MODELLO}")
    print("=" * 60)

    # — STEP 1: Leggi i CSV —
    print("\n📂 Lettura dei file CSV...")
    general_ctx  = leggi_csv(general_csv)
    business_ctx = leggi_csv(business_csv)
    features_ctx = leggi_csv(features_csv)

    # — STEP 2: Primo prompt —
    print("🧠 Invio contesto al modello (potrebbe richiedere qualche secondo)...\n")
    prompt_iniziale = costruisci_prompt_iniziale(
        general_ctx, business_ctx, features_ctx
    )

    # La cronologia mantiene tutta la conversazione in memoria
    cronologia = [{"role": "user", "content": prompt_iniziale}]

    # — STEP 3: Prima risposta —
    risposta = chiedi_a_ollama(cronologia)
    cronologia.append({"role": "assistant", "content": risposta})

    # — STEP 4: Loop di feedback interattivo —
    while True:
        print("\n" + "─" * 60)
        print("📋 COMPRENSIONE CORRENTE DELL'AGENTE:\n")
        print(risposta)
        print("\n" + "─" * 60)
        print(f"\n✏️  Inserisci feedback per affinare (o '{PAROLA_MAGICA}' per confermare):\n")

        feedback = input("Tu → ").strip()

        # Parola magica: esce dal loop
        if feedback.lower() == PAROLA_MAGICA:
            print("\n🎯 Comprensione confermata! Salvataggio in corso...")
            break

        if not feedback:
            print("  [INFO] Nessun testo inserito, riprova.")
            continue

        # Aggiunge il feedback alla cronologia e ottiene nuova risposta
        cronologia.append({"role": "user", "content": feedback})
        print("\n⏳ Aggiornamento comprensione...")
        risposta = chiedi_a_ollama(cronologia)
        cronologia.append({"role": "assistant", "content": risposta})

    # — STEP 5: Estrai JSON e salva —
    print("\n💾 Estrazione struttura finale...")
    struttura = estrai_struttura_json(risposta)
    salva_memoria(struttura)

    # Mostra il risultato finale
    print("\n📦 MEMORIA FINALE:")
    print(json.dumps(struttura, ensure_ascii=False, indent=2))

    return struttura
    



In [13]:
# ─────────────────────────────────────────────
# 8. ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    esegui_agente_contesto(
        general_csv="IF_context_infos/general_context.csv",
        business_csv="IF_context_infos/biz_context.csv",
        features_csv="IF_context_infos/features_context.csv"
    )

  AGENTE CONTESTUALE — powered by Ollama (locale)
  Modello: phi3:mini

📂 Lettura dei file CSV...
🧠 Invio contesto al modello (potrebbe richiedere qualche secondo)...


────────────────────────────────────────────────────────────
📋 COMPRENSIONE CORRENTE DELL'AGENTE:

Agent Role: Explainable AI (XAI) Data Scientist with a focus on delivering Action-Oriented Insights for E-Mobility Sales Strategy Optimization in the Retail Sector to maximize Conversions and Revenues through Personalized Customer Profiles.

General Context Summary:
The domain involves e-mobility retail sales, specifically electric scooters within a B2C business environment where data science plays a pivotal role by providing insights for targeted marketing strategies to enhance customer profiles and personalize the shopping experience. The macro-level dynamics include understanding consumer behaviors such as purchasing power (Capacita_d_spsa), lifetime value, recent transaction patterns, age demographics (Eta/Tenure), dig

Tu →  yes



🎯 Comprensione confermata! Salvataggio in corso...

💾 Estrazione struttura finale...

✅ Memoria salvata in: context_memory.json

📦 MEMORIA FINALE:
{
  "role": "Explainable AI Data Scientist",
  "general_context": "E-Mobility sales strategy optimization for maximizing conversions and revenues through personalized customer profiles, focusing on electric scooters in a B2C retail setting within the e-mobility sector.",
  "business_context": "Identifying high propensity customers using socio-demographic data and purchasing behavior to inform targeted marketing strategies for maximized sales conversions, with an emphasis on personalization based on concrete customer behaviors and preferences without assumptions or extrapolations.",
  "scope": "To develop a detailed understanding of high-impact customer segments using XAI insights aligned with business objectives. The data scientist will identify key features for profiling customers likely to purchase electric scooters, which includes socio-

In [3]:

# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────

OLLAMA_URL    = "http://localhost:11434/api/chat"
MODELLO       = "phi3:mini"
MAGIC_WORD    = "yes"

CONTEXT_MEMORY_FILE = "context_memory.json"
OUTPUT_JSON         = "profile_descriptions.json"
TIMEOUT             = 5000

# All possible column names that identify a profile
# (case-sensitive — add more here if needed)
PROFILE_COLUMNS = ["profile_ID", "Profile_ID", "profile_id", "profile", "Profile",
                   "cluster", "Cluster", "segment", "Segment", "group", "Group"]


# ─────────────────────────────────────────────
# 2. LOAD CONTEXT MEMORY (from first agent)
# ─────────────────────────────────────────────

def load_context_memory(filepath: str = CONTEXT_MEMORY_FILE) -> str:
    """
    Loads the JSON memory produced by the first context agent
    and formats it as readable text for the prompt.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"\n❌ '{filepath}' not found!\n"
            "   Run the context agent first to generate it."
        )
    with open(filepath, "r", encoding="utf-8") as f:
        memory = json.load(f)

    text = f"""
ROLE: {memory.get('ruolo', 'N/A')}
GENERAL CONTEXT: {memory.get('contesto_generale', 'N/A')}
BUSINESS CONTEXT: {memory.get('contesto_business', 'N/A')}
PURPOSE: {memory.get('scopo', 'N/A')}
FEATURES AND CONCEPTS: {', '.join(memory.get('features_e_concetti', []))}
""".strip()

    print(f"  [OK] Context memory loaded from '{filepath}'")
    return text


# ─────────────────────────────────────────────
# 3. READ CSV FILES
# ─────────────────────────────────────────────

def read_csv(filepath: str, sep: str = ";") -> pd.DataFrame:
    """
    Reads a CSV file with configurable separator.
    Tries UTF-8 first, then latin-1 as fallback.
    Returns empty DataFrame if file is not found.
    """
    if not os.path.exists(filepath):
        print(f"  [WARN] File not found: {filepath} — skipped.")
        return pd.DataFrame()

    for enc in ["utf-8", "latin-1"]:
        try:
            df = pd.read_csv(filepath, sep=sep, encoding=enc)
            print(f"  [OK] {filepath} — {df.shape[0]} rows, {df.shape[1]} cols")
            return df
        except Exception:
            continue

    print(f"  [ERR] Cannot read: {filepath}")
    return pd.DataFrame()


# ─────────────────────────────────────────────
# 4. READ SHAP JSON
# ─────────────────────────────────────────────

def read_shap_json(filepath: str) -> str:
    """
    Reads the SHAP importance JSON file and returns it
    as a formatted text string ready for the prompt.
    Returns 'Not available.' if the file is missing or invalid.
    """
    if not os.path.exists(filepath):
        print(f"  [WARN] SHAP file not found: {filepath} — skipped.")
        return "Not available."
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"  [OK] {filepath} — SHAP data loaded.")
        # Convert JSON to readable text for the prompt
        return json.dumps(data, ensure_ascii=False, indent=2)
    except json.JSONDecodeError as e:
        print(f"  [ERR] Cannot parse {filepath}: {e}")
        return "Not available."


# ─────────────────────────────────────────────
# 5. PROFILE UTILITIES
# ─────────────────────────────────────────────

def find_profile_column(df: pd.DataFrame) -> str | None:
    """
    Returns the name of the profile column found in the DataFrame,
    or None if no known profile column exists.
    """
    for col in PROFILE_COLUMNS:
        if col in df.columns:
            return col
    return None


def extract_profiles(df_stats: pd.DataFrame) -> list:
    """
    Extracts the unique list of profile IDs from the stats DataFrame.
    Uses the first matching profile column from PROFILE_COLUMNS.
    Falls back to the first column if none is found.
    """
    col = find_profile_column(df_stats)
    if col:
        profiles = df_stats[col].unique().tolist()
        print(f"  [OK] Profile column found: '{col}' — {len(profiles)} unique profiles")
        return profiles

    fallback_col = df_stats.columns[0]
    print(f"  [WARN] No profile column found, using first column: '{fallback_col}'")
    return df_stats[fallback_col].unique().tolist()


def filter_by_profile(df: pd.DataFrame, profile_id: str) -> str:
    """
    Filters a DataFrame by profile_id and returns data as a clean
    'feature: value' list — much easier for the model to read than
    a flat tabular string with misaligned columns.
    If no profile column exists, returns the full DataFrame as-is
    (used for global data sources like odd ratios).
    """
    if df.empty:
        return "Not available."

    col = find_profile_column(df)
    if col:
        subset = df[df[col].astype(str) == str(profile_id)]
        if not subset.empty:
            # Convert each row to readable "feature: value" lines
            # This avoids misaligned column headers confusing the model
            lines = []
            for _, row in subset.iterrows():
                for colname, value in row.items():
                    if colname != col:  # skip the profile_id column itself
                        lines.append(f"  {colname}: {value}")
                lines.append("")  # blank line between rows
            return "\n".join(lines).strip()

    # No profile column — return full table (global data)
    return df.to_string(index=False)


# ─────────────────────────────────────────────
# 6. CALL OLLAMA
# ─────────────────────────────────────────────

def ask_ollama(messages: list) -> str:
    """
    Sends a message list to Ollama and returns the text response.
    Handles connection errors and timeouts gracefully.
    """
    payload = {
        "model": MODELLO,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": 0.4,   # balanced: creative but precise
            "num_predict": 1000   # max response length in tokens
        }
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
        response.raise_for_status()
        return response.json()["message"]["content"]

    except requests.exceptions.ConnectionError:
        raise ConnectionError(
            "\n❌ Ollama is not running!\n"
            "   Start it with: ollama serve"
        )
    except requests.exceptions.Timeout:
        raise TimeoutError(
            f"\n❌ Timeout after {TIMEOUT}s.\n"
            "   Try reducing CSV data size or increase TIMEOUT."
        )


# ─────────────────────────────────────────────
# 7. FEEDBACK LOOP
# ─────────────────────────────────────────────

def feedback_loop(initial_response: str, history: list, label: str) -> str:
    """
    Shows the current model response and collects user feedback.
      - Free text  → refines the response (new call to Ollama)
      - "yes"      → confirms and exits the loop
    Returns the final confirmed response.
    """
    response = initial_response

    while True:
        print(f"\n{'─' * 60}")
        print(f"📋 {label}:\n")
        print(response)
        print(f"\n{'─' * 60}")
        print(f"\n✏️  Give feedback to refine, or type '{MAGIC_WORD}' to confirm:\n")

        user_input = input("You → ").strip()

        if user_input.lower() == MAGIC_WORD:
            print("✅ Confirmed!")
            return response

        if not user_input:
            print("  [INFO] Empty input, try again.")
            continue

        history.append({"role": "user", "content": user_input})
        print("\n⏳ Updating response...")
        response = ask_ollama(history)
        history.append({"role": "assistant", "content": response})


# ─────────────────────────────────────────────
# 8. BUILD PROFILE PROMPT
# ─────────────────────────────────────────────

def build_profile_prompt(
    context:   str,
    profile_id: str,
    stats:     str,
    shap:      str,
    features:  str,
    oddratio:  str,
    rules:     str
) -> str:
    """
    Builds the full prompt for a single profile.
    The profile_id is injected explicitly at the top so the model
    never has to guess or search for it in the tabular data.
    """
    return f"""
You are an expert data analyst and storyteller specialized in machine learning model output profiling.
You analyze and make actionable profiles based on ML model score.
You write compelling, data-driven, actionable profile descriptions for stakeholders.

---
## BUSINESS CONTEXT
{context}

---
## ⚠️ YOU ARE ANALYZING EXACTLY THIS PROFILE:
# Profile ID = {profile_id}
# This ID comes directly from the profile_stats file.
# Use it as-is. Do not change it, do not invent others.

---
## DATA FOR PROFILE ID: {profile_id}

### Statistical Summary (mean and mode per feature and Profile_ID)
{stats}

### SHAP Mean Values (average feature impact on model prediction)
{shap}

### Feature Importance (decision tree — top drivers for this profile)
{features}

### Odd Ratios (logistic regression — likelihood vs average population)
{oddratio}

### Assignment Rules (conditions that assign a customer to this profile)
{rules}

---
## YOUR TASK

Write a complete description of the profile with ID = {profile_id}.
Use ONLY the numbers and values listed above. Do NOT invent data.

Structure your answer as follows:

---
### 🎯 Profile Name & Tagline | ID: {profile_id}
- **Name**: A short memorable name (e.g. "The Loyal High-Spender")
- **Tagline**: One sentence capturing who they are

---
### 📖 Who Are They? (Storytelling)
3-5 sentences. Tell a vivid human story using the actual feature values above.
Ground every sentence in real data — e.g. "On average, they spend X per month and visit Y times."

---
### 📊 Key Data Insights
2-5 bullet points in this format:
- **[feature name]** = [value from data] → [what this means for the business]
Only use values explicitly present in the data above.

---
### ⚡ Actionable Recommendations
Provide 2-5 concrete, specific actions. Each must:
- Start with a verb (e.g. "Launch", "Target", "Reduce", "Increase")
- Reference specific data values to justify the action
- Be directly executable by a business team

---
### ⚠️ Watch Out For
List 1-2 risks or pitfalls specific to this profile based on the data.

---
Be specific. Be data-driven. Avoid generic statements like "this segment is important".
Every claim must trace back to a number or pattern in the data provided.
""".strip()


# ─────────────────────────────────────────────
# 9. ANALYZE A SINGLE PROFILE
# ─────────────────────────────────────────────

def analyze_profile(
    context:     str,
    profile_id:  str,
    shap_txt:    str,
    df_stats:    pd.DataFrame,
    df_features: pd.DataFrame,
    df_oddratio: pd.DataFrame,
    df_rules:    pd.DataFrame
) -> dict:
    """
    Generates the full description for a single profile:
      1. Filters each DataFrame for this profile_id
      2. Builds the prompt with all data + explicit profile_id
      3. Calls Ollama
      4. Opens feedback loop until user types "yes"
      5. Returns the confirmed description as a dict
    """
    print(f"\n  🔍 Analyzing profile: '{profile_id}'...")

    # Filter each data source for this specific profile
    stats_txt    = filter_by_profile(df_stats,    profile_id)
    features_txt = filter_by_profile(df_features, profile_id)
    oddratio_txt = filter_by_profile(df_oddratio, profile_id)
    rules_txt    = filter_by_profile(df_rules,    profile_id)

    # Build prompt — profile_id injected explicitly
    prompt = build_profile_prompt(
        context, profile_id,
        stats_txt, shap_txt, features_txt, oddratio_txt, rules_txt
    )

    # First model call
    history  = [{"role": "user", "content": prompt}]
    response = ask_ollama(history)
    history.append({"role": "assistant", "content": response})

    # Feedback loop — user can refine or confirm with "yes"
    final_description = feedback_loop(
        response, history,
        label=f"PROFILE DESCRIPTION — ID: {profile_id}"
    )

    return {
        "profile_id":  str(profile_id),
        "description": final_description
    }


# ─────────────────────────────────────────────
# 10. SAVE OUTPUT
# ─────────────────────────────────────────────

def save_output(results: list, filepath: str = OUTPUT_JSON):
    """Saves all profile descriptions to a structured JSON file."""
    output = {
        "total_profiles": len(results),
        "profiles": results
    }
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"\n✅ All profile descriptions saved to: {filepath}")


# ─────────────────────────────────────────────
# 11. MAIN FUNCTION
# ─────────────────────────────────────────────

def run_profile_agent(
    oddratio_csv:         str = "oddsratio.csv",
    profile_features_csv: str = "profile_features.csv",
    profile_stats_csv:    str = "profile_stats.csv",
    profile_rules_csv:    str = "profile_rules.csv",
    shap_importance:      str = "shap_importance.json",
    context_memory:       str = CONTEXT_MEMORY_FILE,
    sep:                  str = ";"
):
    """
    Main function of the profile analyst agent:
      1. Loads context memory from first agent
      2. Reads the profile CSVs + SHAP JSON
      3. Extracts unique profile IDs from profile_stats.csv
      4. For each profile: generates description → feedback loop → "yes"
      5. Saves everything to profile_descriptions.json
    """
    print("=" * 60)
    print("  PROFILE ANALYST AGENT — powered by Ollama phi3:mini")
    print("=" * 60)

    # STEP 1 — Load context from first agent
    print("\n📚 Loading business context memory...")
    context = load_context_memory(context_memory)

    # STEP 2 — Read profile CSVs
    print("\n📂 Loading profile CSV files...")
    df_oddratio  = read_csv(oddratio_csv,         sep)
    df_features  = read_csv(profile_features_csv, sep)
    df_stats     = read_csv(profile_stats_csv,    sep)
    df_rules     = read_csv(profile_rules_csv,    sep)

    # STEP 3 — Read SHAP JSON (global, not filtered per profile)
    print("\n📂 Loading SHAP JSON...")
    shap_txt = read_shap_json(shap_importance)

    # STEP 4 — Extract profile list from stats (most complete source)
    if df_stats.empty:
        raise ValueError(
            "❌ profile_stats.csv is empty or not found.\n"
            "   This file is required to extract the profile list."
        )
    profiles = extract_profiles(df_stats)
    print(f"\n🎯 Profiles to analyze: {profiles}")
    print(f"   Total: {len(profiles)}")

    # STEP 5 — Analyze each profile with feedback loop
    results = []
    for i, profile_id in enumerate(profiles, 1):
        print(f"\n{'█' * 60}")
        print(f"  PROFILE {i} of {len(profiles)}: '{profile_id}'")
        print(f"{'█' * 60}")

        result = analyze_profile(
            context, profile_id, shap_txt,
            df_stats, df_features, df_oddratio, df_rules
        )
        results.append(result)
        print(f"\n✅ Profile '{profile_id}' confirmed — moving to next.")

    # STEP 6 — Save all results
    save_output(results)

    print(f"\n🎉 Done! {len(results)} profiles analyzed.")
    print(f"   Output: {OUTPUT_JSON}")

    return results


In [ ]:

# ─────────────────────────────────────────────
# 12. ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    run_profile_agent(
        oddratio_csv         = "IF_analytical_output/oddsratio.csv",
        profile_features_csv = "IF_analytical_output/profile_features.csv",
        profile_stats_csv    = "IF_analytical_output/profile_stats.csv",
        profile_rules_csv    = "IF_analytical_output/profile_rules.csv",
        shap_importance      = "shap_importance.json",
        context_memory       = "context_memory.json",
        sep                  = ","
    )


  PROFILE ANALYST AGENT — powered by Ollama phi3:mini

📚 Loading business context memory...
  [OK] Context memory loaded from 'context_memory.json'

📂 Loading profile CSV files...
  [OK] IF_analytical_output/oddsratio.csv — 31 rows, 7 cols
  [OK] IF_analytical_output/profile_features.csv — 10 rows, 2 cols
  [OK] IF_analytical_output/profile_stats.csv — 7 rows, 28 cols
  [OK] IF_analytical_output/profile_rules.csv — 5 rows, 2 cols

📂 Loading SHAP JSON...
  [OK] shap_importance.json — SHAP data loaded.
  [OK] Profile column found: 'Profile_ID' — 7 unique profiles

🎯 Profiles to analyze: ['23', '5', '6', '7', '8', 'Generic', 'NotHP']
   Total: 7

████████████████████████████████████████████████████████████
  PROFILE 1 of 7: '23'
████████████████████████████████████████████████████████████

  🔍 Analyzing profile: '23'...

────────────────────────────────────────────────────────────
📋 PROFILE DESCRIPTION — ID: 23:

### 🎯 Profile Name & Tagline | ID: 23
- **Name**: The Eco-Conscious Digital 

You →  yes


✅ Confirmed!

✅ Profile '23' confirmed — moving to next.

████████████████████████████████████████████████████████████
  PROFILE 2 of 7: '5'
████████████████████████████████████████████████████████████

  🔍 Analyzing profile: '5'...

────────────────────────────────────────────────────────────
📋 PROFILE DESCRIPTION — ID: 5:

### 🎯 Profile Name & Tagline | ID: 5
- **Name**: The Loyal Electric Bike Purchaser - Operaio Lifecyle Sinker
- **Tagline**: Operators with a deep connection to traditional work and eco-friendly choices.

### 📖 Who Are They? (Storytelling)
Operaio, representing the manual labor professionals in our customer base, show their loyalty through consistent purchases of electric bicycles within the last year with an average spend reflecting a commitment to sustainable mobility. With an RFM score suggesting high recency and frequency, these customers are deeply engaged at 65%, likely valuing both community interaction (as indicated by engagement) and digital savvy capabilit

You →  yes


✅ Confirmed!

✅ Profile '5' confirmed — moving to next.

████████████████████████████████████████████████████████████
  PROFILE 3 of 7: '6'
████████████████████████████████████████████████████████████

  🔍 Analyzing profile: '6'...

────────────────────────────────────────────────────────────
📋 PROFILE DESCRIPTION — ID: 6:

### Profile Name & Tagline | ID: 6
- **Name**: The Loyal Green Electricians (Eco-Conscious Cyclists)
- **Tagline**: Highly engaged, eco-friendly working professionals with a penchant for cycling and smart spending.

### 📖 Who Are They? (Storytelling)
The Loyal Green Electricians represent our most environmentally aware customers who have made an average monthly spend of $499 on protective gear, reflecting a strong commitment to safety and sustainability. With over half the year having passed since their last purchase—a new electric bike—they demonstrate brand loyalty with repeat visits every 2 months (1.9 times). Their digital savvy is evident as they're among our t

...

✅ Confirmed!

✅ Profile '6' confirmed — moving to next.

████████████████████████████████████████████████████████████
  PROFILE 4 of 7: '7'
████████████████████████████████████████████████████████████

  🔍 Analyzing profile: '7'...

────────────────────────────────────────────────────────────
📋 PROFILE DESCRIPTION — ID: 7:

### 🎯 Profile Name & Tagline | ID: 7
- **Name**: The Industrious Employee Guardian Guanti (IEG)
- **Tagline**: Average spenders with high loyalty and a green heart, primarily purchasing protective gear.

### 📖 Who Are They? (Storytelling)
The IEG segment represents employees in their working years who have an average Lifetime Value of $3.04k over the last year with moderate engagement and digitalization levels, indicating a steady customer base that values practicality alongside environmental consciousness. On average, they spend 2 guanti per month within this profile's lifecycle span and visit our platform at least once every three months, suggesting regular intera

You →  yes but know stop analyzing profiles produce directly the final output wihtout doing other analysis 



⏳ Updating response...

────────────────────────────────────────────────────────────
📋 PROFILE DESCRIPTION — ID: 7:

### Final Profile Description for ID: 7 - The Industrious Employee Guardian Guanti (IEG)

---
#### Name & Tagline: A short memorable name and a one-sentence capturing who they are.
- **Name**: Guardians of the Workplace – IEG
- **Tagline**: Employees in their working years, with an average LTV of $3.04k over the last year, show loyalty to safety gear and environmental consciousness through consistent purchases like guanti.

#### Who Are They? (Storytelling)
The Guardians are employees aged 35 who have spent on protective equipment such as gloves with an average frequency of once every three months over the past year, reflecting a steady but not excessive engagement pattern within their age demographic and lifecycle stage. Their GreenSurveyScore stands at 8.47, indicating that they are highly environmentally conscious consumers who prefer to make purchases like guanti wi

You →  yes


✅ Confirmed!

✅ Profile '7' confirmed — moving to next.

████████████████████████████████████████████████████████████
  PROFILE 5 of 7: '8'
████████████████████████████████████████████████████████████

  🔍 Analyzing profile: '8'...
